In [146]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
project_dir = Path(".").resolve().parents[0] # sobe da pasta notebooks para a raiz do projeto
data_raw_dir = project_dir / "data" / "raw"

project_dir, data_raw_dir


(PosixPath('/Users/lucas/Documents/Projects/olist-analytics'),
 PosixPath('/Users/lucas/Documents/Projects/olist-analytics/data/raw'))

In [147]:
orders = pd.read_csv(data_raw_dir / "olist_orders_dataset.csv")
order_items = pd.read_csv(data_raw_dir / "olist_order_items_dataset.csv")
customers = pd.read_csv(data_raw_dir / "olist_customers_dataset.csv")


## 1. Orders Dataset

Tabela central do dataset. Todo pedido passa por aqui antes de se conectar
com itens, clientes e pagamentos. Vou inspecionar estrutura, tipos e nulos.

In [148]:
orders.head(10)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00
5,a4591c265e18cb1dcee52889e2d8acc3,503740e9ca751ccdda7ba28e9ab8f608,delivered,2017-07-09 21:57:05,2017-07-09 22:10:13,2017-07-11 14:58:04,2017-07-26 10:57:55,2017-08-01 00:00:00
6,136cce7faa42fdb2cefd53fdc79a6098,ed0271e0b7da060a393796590e7b737a,invoiced,2017-04-11 12:22:08,2017-04-13 13:25:17,NaN,NaN,2017-05-09 00:00:00
7,6514b8ad8028c9f2cc2374ded245783f,9bdf08b4b3b52b5526ff42d37d47f222,delivered,2017-05-16 13:10:30,2017-05-16 13:22:11,2017-05-22 10:07:46,2017-05-26 12:55:51,2017-06-07 00:00:00
8,76c6e866289321a7c93b82b54852dc33,f54a9f0e6b351c431402b8461ea51999,delivered,2017-01-23 18:29:09,2017-01-25 02:50:47,2017-01-26 14:16:31,2017-02-02 14:08:10,2017-03-06 00:00:00
9,e69bfb5eb88e0ed6a785585b27e16dbf,31ad1d1b63eb9962463f764d4e6e0c9d,delivered,2017-07-29 11:55:02,2017-07-29 12:05:32,2017-08-10 19:45:24,2017-08-16 17:14:30,2017-08-23 00:00:00


### Observações - head()
- 8 colunas, todas relacionadas ao ciclo de vida do pedido (compra → aprovação → entrega)
- order_id e customer_id são hashes — não têm valor numérico, só servem como chave
- Há timestamps em 5 colunas diferentes, o que indica que será possível calcular
  tempo de aprovação, tempo de entrega e atraso em relação ao prazo estimado
- Na linha 6 (índice 6), order_status = "invoiced" e duas colunas de entrega estão NaN
  — faz sentido, pedido faturado mas não entregue ainda

In [149]:
orders.info()


<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   order_id                       99441 non-null  str  
 1   customer_id                    99441 non-null  str  
 2   order_status                   99441 non-null  str  
 3   order_purchase_timestamp       99441 non-null  str  
 4   order_approved_at              99281 non-null  str  
 5   order_delivered_carrier_date   97658 non-null  str  
 6   order_delivered_customer_date  96476 non-null  str  
 7   order_estimated_delivery_date  99441 non-null  str  
dtypes: str(8)
memory usage: 6.1 MB


### Observações - info()
- 99.441 pedidos no total
- Todos os timestamps estão como string (object/str), não datetime — precisará de conversão
- order_approved_at: 160 nulos (99441 - 99281)
- order_delivered_carrier_date: 1783 nulos — pedidos não enviados ainda
- order_delivered_customer_date: 2965 nulos — pedidos não entregues
- Esses nulos são esperados para pedidos cancelados ou em andamento
  

In [150]:
orders.describe()


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
count,99441,99441,99441,99441,99281,97658,96476,99441
unique,99441,99441,8,98875,90733,81018,95664,459
top,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2018-03-31 15:08:21,2018-02-27 04:31:10,2018-05-09 15:48:00,2018-05-14 20:02:44,2017-12-20 00:00:00
freq,1,1,96478,3,9,47,3,522


### Observações - describe()
- order_status tem 8 categorias únicas, com "delivered" dominando (96.478 de 99.441 = ~97%)
- order_estimated_delivery_date tem apenas 459 valores únicos para 99.441 pedidos
  — a Olist agrupa estimativas por lotes, não calcula individualmente para cada pedido

## 2. Order Items Dataset


In [151]:
order_items.head(10)

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14
5,00048cc3ae777c65dbb7d2a0634bc1ea,1,ef92defde845ab8450f9d70c526ef70f,6426d21aca402a131fc0a5d0960a3c90,2017-05-23 03:55:27,21.90,12.69
6,00054e8431b9d7675808bcb819fb4a32,1,8d4f2bb7e93e6710a28f34fa83ee7d28,7040e82f899a04d1b434b795a43b4617,2017-12-14 12:10:31,19.90,11.85
7,000576fe39319847cbb9d288c5617fa6,1,557d850972a7d6f792fd18ae1400d9b6,5996cddab893a4652a15592fb58ab8db,2018-07-10 12:30:45,810.00,70.75
8,0005a1a1728c9d785b8e2b08b904576c,1,310ae3c140ff94b03219ad0adc3c778f,a416b6a846a11724393025641d4edd5e,2018-03-26 18:31:29,145.95,11.65
9,0005f50442cb953dcd1d21e1fb923495,1,4535b0e1091c278dfd193e5a1d63b39f,ba143b05f0110f0dc71ad71b4466ce92,2018-07-06 14:10:56,53.99,11.40


### Observações - head()
- 7 colunas: conecta pedido → produto → vendedor, com preço e frete
- order_item_id indica a posição do item dentro do pedido (1 = primeiro item)
- Nos primeiros 10 registros todos são item 1 — a maioria dos pedidos tem 1 item só
- price e freight_value são os únicos campos numéricos "reais"

In [152]:
order_items.info()

<class 'pandas.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   order_id             112650 non-null  str    
 1   order_item_id        112650 non-null  int64  
 2   product_id           112650 non-null  str    
 3   seller_id            112650 non-null  str    
 4   shipping_limit_date  112650 non-null  str    
 5   price                112650 non-null  float64
 6   freight_value        112650 non-null  float64
dtypes: float64(2), int64(1), str(4)
memory usage: 6.0 MB


### Observações - info()
- 112.650 linhas vs 99.441 pedidos — a diferença indica pedidos com múltiplos itens
- Nenhum valor nulo em nenhuma coluna
- price e freight_value já estão como float64 — não precisam de conversão
- shipping_limit_date está como string — precisará de conversão para datetime


In [153]:
order_items.describe()

,order_item_id,price,freight_value
count,112650.000000,112650.000000,112650.000000
mean,1.197834,120.653739,19.990320
std,0.705124,183.633928,15.806405
min,1.000000,0.850000,0.000000
25%,1.000000,39.900000,13.080000
50%,1.000000,74.990000,16.260000
75%,1.000000,134.900000,21.150000
max,21.000000,6735.000000,409.680000


### Observações - describe()
- order_item_id: média 1.19 e 75% no valor 1 — confirma que a maioria dos pedidos tem só 1 item, mas o máximo é 21
- price: mediana R$74,99 mas média R$120,65 — distribuição assimétrica, provável cauda de itens caros
- freight_value: min 0 (frete grátis existe), máximo R$409,68

## 3. Customers Dataset


In [154]:
customers.head(10)

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP
5,879864dab9bc3047522c92c82e1212b8,4c93744516667ad3b8f1fb645a3116a4,89254,jaragua do sul,SC
6,fd826e7cf63160e536e0908c76c3f441,addec96d2e059c80c30fe6871d30d177,4534,sao paulo,SP
7,5e274e7a0c3809e14aba7ad5aae0d407,57b2a98a409812fe9618067b6b8ebe4f,35182,timoteo,MG
8,5adf08e34b2e993982a47070956c5c65,1175e95fb47ddff9de6b2b06188f7e0d,81560,curitiba,PR
9,4b7139f34592b3a31687243a302fa75b,9afe194fb833f79e300e37e580171f22,30575,belo horizonte,MG


### Observações - head()
- 5 colunas: customer_id, customer_unique_id, zip_code_prefix, city e state
- customer_id é a chave de ligação com orders — é por pedido
- customer_unique_id identifica o cliente real — um mesmo cliente pode ter vários customer_ids
- Nos primeiros registros predominância de SP, o que faz sentido pelo tamanho do estado

In [155]:
customers.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   customer_id               99441 non-null  str  
 1   customer_unique_id        99441 non-null  str  
 2   customer_zip_code_prefix  99441 non-null  int64
 3   customer_city             99441 non-null  str  
 4   customer_state            99441 non-null  str  
dtypes: int64(1), str(4)
memory usage: 3.8 MB


### Observações - info()
- 99.441 linhas — mesmo número de pedidos, confirmando 1 registro por pedido
- Sem nulos em nenhuma coluna
- customer_zip_code_prefix está como int64 — tecnicamente é um código, não um número

In [156]:
customers.describe()

,customer_zip_code_prefix
count,99441.000000
mean,35137.474583
std,29797.938996
min,1003.000000
25%,11347.000000
50%,24416.000000
75%,58900.000000
max,99990.000000


### Observações - describe()
- Só o zip_code_prefix aparece no describe() pois é o único campo numérico
- Estatísticas de CEP não têm significado analítico real — é só um identificador

## 4. Integração dos Dados — Merges

Dois DataFrames unificados foram criados para suportar análises diferentes:
- `delivery_order`: orders + customers — usado para análise de entrega
- `order_completed`: orders + order_items + customers — usado para todas as outras análises

In [157]:
delivery_order = orders.merge(customers, on='customer_id')


In [158]:
order_complet = orders.merge(order_items, on='order_id')
order_completed = order_complet.merge(customers, on='customer_id')


## 5. Análise Exploratória

### 5.1 Volume de Pedidos por Estado

In [159]:
order_completed.value_counts('customer_state')


customer_state
SP    47449
RJ    14579
MG    13129
RS     6235
PR     5740
SC     4176
BA     3799
DF     2406
GO     2333
ES     2256
PE     1806
CE     1478
PA     1080
MT     1055
MA      824
MS      819
PB      602
PI      542
RN      529
AL      444
SE      385
TO      315
RO      278
AM      165
AC       92
AP       82
RR       52
Name: count, dtype: int64

### Observações — Volume por Estado
- SP com 47.449 pedidos domina de forma clara — faz sentido pelo tamanho do estado
- RJ e MG em segundo e terceiro, bem distantes de SP
- Estados do Norte aparecem com volume muito baixo — não sei se é falta de acesso, logística ruim ou simplesmente menos usuários
- Essa concentração no Sudeste provavelmente vai aparecer em outras análises também

### 5.2 Ticket Médio por Estado

Para evitar inflar o ticket por pedidos com múltiplos itens, o preço foi
somado por pedido primeiro e depois calculada a média por estado.

In [160]:
result_group = order_completed.groupby(['order_id', 'customer_state'])['price'].sum()
result_group.groupby('customer_state').mean().sort_values(ascending=False)


customer_state
PB    216.669323
AP    198.151471
AC    197.320370
AL    195.413163
RO    186.804211
PA    184.482278
TO    177.855699
PI    176.296308
MT    173.259723
RN    172.271743
CE    171.254491
SE    170.785072
RR    170.205000
MS    164.756897
MA    161.686784
PE    159.458756
BA    152.278139
AM    152.087347
GO    146.782237
SC    144.117757
RJ    142.931568
DF    142.401854
RS    138.126661
MG    137.327445
PR    136.671421
ES    135.820894
SP    125.751179
Name: price, dtype: float64

### Observações — Ticket Médio por Estado
- Achei interessante: os estados com menor volume (Norte e Nordeste) têm ticket médio mais alto
- Uma hipótese que surgiu: quem compra nessas regiões talvez escolha produtos mais caros porque o frete já é alto de qualquer forma
- SP, que lidera em volume, aparece na parte de baixo do ranking — muitos pedidos baratos puxam a média pra baixo
- Não tenho certeza se essa interpretação está correta — vale investigar mais

### 5.3 Pedidos por Mês

`order_purchase_timestamp` foi convertido para datetime antes de extrair o período.

In [161]:
order_completed['order_purchase_timestamp'] = pd.to_datetime(order_completed['order_purchase_timestamp'])


In [162]:
order_completed['order_purchase_timestamp'].dt.to_period('M').value_counts()


order_purchase_timestamp
2017-11    8665
2018-03    8217
2018-01    8208
2018-04    7975
2018-05    7925
2018-02    7672
2018-08    7248
2018-07    7092
2018-06    7078
2017-12    6308
2017-10    5322
2017-08    4910
2017-09    4831
2017-07    4519
2017-05    4136
2017-06    3583
2017-03    3000
2017-04    2684
2017-02    1951
2017-01     955
2016-10     363
2016-09       6
2016-12       1
2018-09       1
Freq: M, Name: count, dtype: int64

### Observações — Pedidos por Mês
- Novembro de 2017 foi o pico com 8.665 pedidos — provavelmente Black Friday
- Dá pra ver um crescimento ao longo de 2017, o que indica que o negócio estava expandindo
- 2016 tem pouquíssimos registros — o dataset deve começar no meio do ano
- A partir de 2018 o volume parece se estabilizar, mas o dataset pode estar cortado antes do fim do ano

In [163]:
order_completed.info()


<class 'pandas.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 18 columns):
 #   Column                         Non-Null Count   Dtype         
---  ------                         --------------   -----         
 0   order_id                       112650 non-null  str           
 1   customer_id                    112650 non-null  str           
 2   order_status                   112650 non-null  str           
 3   order_purchase_timestamp       112650 non-null  datetime64[us]
 4   order_approved_at              112635 non-null  str           
 5   order_delivered_carrier_date   111456 non-null  str           
 6   order_delivered_customer_date  110196 non-null  str           
 7   order_estimated_delivery_date  112650 non-null  str           
 8   order_item_id                  112650 non-null  int64         
 9   product_id                     112650 non-null  str           
 10  seller_id                      112650 non-null  str           
 11  shipping_li

### 5.4 Média de Itens por Pedido

In [164]:
unique_order = order_completed['order_id'].nunique()
mean_order_items = len(order_completed) / unique_order
print(mean_order_items)


1.1417306873695092


### Observações — Média de Itens por Pedido
- 1,14 itens por pedido em média — as pessoas compram praticamente um produto por vez
- Faz sentido pra um marketplace: o cliente busca algo específico, não faz 'carrinho cheio'
- Curiosidade: qual seria o comportamento de clientes recorrentes? Compram mais itens por pedido?

### 5.5 Proporção do Frete

Criada a coluna `freight_ratio` (frete / preço) para medir o custo relativo
do frete em relação ao valor do produto.

In [165]:
order_completed['freight_ratio'] = order_completed['freight_value'] / order_completed['price'] 
order_completed.describe()


,order_purchase_timestamp,order_item_id,price,freight_value,customer_zip_code_prefix,freight_ratio
count,112650,112650.000000,112650.000000,112650.000000,112650.000000,112650.000000
mean,2018-01-01 00:09:48.464376,1.197834,120.653739,19.990320,35119.309090,0.320864
min,2016-09-04 21:15:19,1.000000,0.850000,0.000000,1003.000000,0.000000
25%,2017-09-13 19:17:04,1.000000,39.900000,13.080000,11310.000000,0.134034
50%,2018-01-19 23:02:16,1.000000,74.990000,16.260000,24340.000000,0.231356
75%,2018-05-04 17:30:36.750000,1.000000,134.900000,21.150000,59028.750000,0.393036
max,2018-09-03 09:06:57,21.000000,6735.000000,409.680000,99990.000000,26.235294
std,NaN,0.705124,183.633928,15.806405,29866.120801,0.349894


### Observações — Proporção do Frete
- Criei essa coluna pra entender se o frete é caro em relação ao produto
- Um ratio de 0,5 por exemplo significa que o frete custa metade do produto — o que parece bastante
- Ainda não analisei a distribuição disso — fica como próximo passo cruzar com estados e categorias de produto

### 5.6 Performance de Entrega

Conversão dos timestamps de entrega para datetime para permitir comparação de datas.

In [166]:
colums_delivery = ['order_delivered_customer_date', 'order_estimated_delivery_date']

for colums in colums_delivery:
    order_completed[colums] = pd.to_datetime(order_completed[colums])


### 5.7 Taxa de Entrega no Prazo

In [167]:
order_completed['is_on_time'] = order_completed['order_delivered_customer_date'] <= order_completed['order_estimated_delivery_date']
order_completed['is_on_time'].value_counts(normalize=True) * 100


is_on_time
True     90.08522
False     9.91478
Name: proportion, dtype: float64

### Observações — Taxa de Entrega no Prazo
- 90% no prazo parece bom, mas quase 10% de atraso em volume absoluto é bastante coisa
- Importante: pedidos sem data de entrega (NaN) não entraram nesse cálculo — são pedidos cancelados ou em andamento
- Fica a dúvida: como a Olist define o prazo estimado? Se o prazo já é folgado, 90% pode ser menos impressionante do que parece

### 5.8 Atrasos por Estado

Filtrados apenas os pedidos atrasados usando `~is_on_time`, depois calculados
a contagem absoluta e o percentual do total de atrasos por estado.

In [168]:
delayed_orders = order_completed[~order_completed['is_on_time']]

delay_counts = delayed_orders['customer_state'].value_counts()
delay_pct = (delayed_orders['customer_state'].value_counts(normalize=True) * 100).round(2)

delay_summary = pd.DataFrame({
    'total_delays': delay_counts,
    'percentage_of_total_delays': delay_pct
})
delay_summary = delay_summary.reset_index()

delay_summary.head(10)


,customer_state,total_delays,percentage_of_total_delays
0,SP,3685,32.99
1,RJ,2268,20.31
2,MG,916,8.20
3,BA,620,5.55
4,RS,524,4.69
5,SC,472,4.23
6,PR,361,3.23
7,ES,303,2.71
8,CE,270,2.42
9,PE,238,2.13


### Observações — Atrasos por Estado
- SP lidera em atrasos absolutos mas também tem o maior volume — não dá pra concluir muito só com esse número
- RJ chama atenção: tem volume bem menor que SP mas aparece com 20% dos atrasos totais
- **Próximo passo**: calcular a taxa de atraso por estado (atrasos / total de pedidos do estado) — só assim dá pra comparar estados de forma justa

### 5.9 Pedidos Entregues por Estado

Filtrei só os pedidos com status "delivered" usando .loc e agrupei por estado pra ver quantas entregas confirmadas cada um tem.

In [169]:
orders_delivereds = order_completed.loc[order_completed['order_status'] == 'delivered']
orders_delivereds.groupby('customer_state')['order_status'].value_counts()


customer_state  order_status
AC              delivered          91
AL              delivered         427
AM              delivered         163
AP              delivered          81
BA              delivered        3683
CE              delivered        1426
DF              delivered        2355
ES              delivered        2225
GO              delivered        2277
MA              delivered         800
MG              delivered       12916
MS              delivered         811
MT              delivered        1037
PA              delivered        1054
PB              delivered         586
PE              delivered        1746
PI              delivered         523
PR              delivered        5649
RJ              delivered       14143
RN              delivered         521
RO              delivered         273
RR              delivered          46
RS              delivered        6134
SC              delivered        4097
SE              delivered         375
SP              deliv

### 5.10 Seleção por Posição — iloc

Testei o .iloc selecionando por posição: primeiras 5 linhas, linhas 10 a 20 e uma célula específica por linha e coluna.

In [170]:
order_completed.iloc[:5, :]


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,freight_ratio,is_on_time
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,1,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,2017-10-06 11:07:15,29.99,8.72,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,0.290764,True
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,1,595fac2a385ac33a80bd5114aec74eb8,289cdb325fb7e7f891c38608bf9e0962,2018-07-30 03:24:27,118.70,22.76,af07308b275d755c9edb36a90c618231,47813,barreiras,BA,0.191744,True
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,1,aa4383b373c6aca5d8797843e5594415,4869f7a5dfa277a7dca6462dcf3b52b2,2018-08-13 08:55:23,159.90,19.22,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO,0.120200,True
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,1,d0b61bfb1de832b15ba9d266ca96e5b0,66922902710d126a0e7d26b0e3805106,2017-11-23 19:45:59,45.00,27.20,7c142cf63193a1473d2e66489a9ae977,59296,sao goncalo do amarante,RN,0.604444,True
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,1,65266b2da20d04dbe00c5c2d3bb7859e,2c9e548be18521d1c43cde1c582c6de8,2018-02-19 20:31:37,19.90,8.72,72632f0f9dd73dfee390c9b22eb56dd6,9195,santo andre,SP,0.438191,True


In [171]:
order_completed.iloc[10:21]


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,freight_ratio,is_on_time
10,e6ce16cb79ec1d90b1da9085a6118aeb,494dded5b201313c64ed7f100595b95c,delivered,2017-05-16 19:41:10,2017-05-16 19:50:18,2017-05-18 11:40:40,2017-05-29 11:18:31,2017-06-07,1,08574b074924071f4e201e151b152b4e,001cca7ae9ae17fb1caed9dfb1094831,2017-05-22 19:50:18,99.00,30.53,f2a85dec752b8517b5e58a06ff3cd937,20780,rio de janeiro,RJ,0.308384,True
11,e6ce16cb79ec1d90b1da9085a6118aeb,494dded5b201313c64ed7f100595b95c,delivered,2017-05-16 19:41:10,2017-05-16 19:50:18,2017-05-18 11:40:40,2017-05-29 11:18:31,2017-06-07,2,08574b074924071f4e201e151b152b4e,001cca7ae9ae17fb1caed9dfb1094831,2017-05-22 19:50:18,99.00,30.53,f2a85dec752b8517b5e58a06ff3cd937,20780,rio de janeiro,RJ,0.308384,True
12,34513ce0c4fab462a55830c0989c7edb,7711cf624183d843aafe81855097bc37,delivered,2017-07-13 19:58:11,2017-07-13 20:10:08,2017-07-14 18:43:29,2017-07-19 14:04:48,2017-08-08,1,f7e0fa615b386bc9a8b9eb52bc1fff76,87142160b41353c4e5fca2360caf6f92,2017-07-19 20:10:08,98.00,16.13,782987b81c92239d922aa49d6bd4200b,4278,sao paulo,SP,0.164592,True
13,82566a660a982b15fb86e904c8d32918,d3e3b74c766bc6214e0c830b17ee2341,delivered,2018-06-07 10:06:19,2018-06-09 03:13:12,2018-06-11 13:29:00,2018-06-19 12:05:52,2018-07-18,1,72a97c271b2e429974398f46b93ae530,094ced053e257ae8cae57205592d6712,2018-06-18 03:13:12,31.90,18.23,e97109680b052ee858d93a539597bba7,35400,ouro preto,MG,0.571473,True
14,5ff96c15d0b717ac6ad1f3d77225a350,19402a48fe860416adf93348aba37740,delivered,2018-07-25 17:44:10,2018-07-25 17:55:14,2018-07-26 13:16:00,2018-07-30 15:52:25,2018-08-08,1,10adb53d8faa890ca7c2f0cbcb68d777,1900267e848ceeba8fa32d80c1a5f5a8,2018-07-27 17:55:14,19.90,12.80,e2dfa3127fedbbca9707b36304996dab,4812,sao paulo,SP,0.643216,True
15,432aaf21d85167c2c86ec9448c4e42cc,3df704f53d3f1d4818840b34ec672a9f,delivered,2018-03-01 14:14:28,2018-03-01 15:10:47,2018-03-02 21:09:20,2018-03-12 23:36:26,2018-03-21,1,72d3bf1d3a790f8874096fcf860e3eff,0bae85eb84b9fb3bd773911e89288d54,2018-03-07 15:10:47,38.25,16.11,04cf8185c71090d28baa4407b2e6d600,5271,sao paulo,SP,0.421176,True
16,dcb36b511fcac050b97cd5c05de84dc3,3b6828a50ffe546942b7a473d70ac0fc,delivered,2018-06-07 19:03:12,2018-06-12 23:31:02,2018-06-11 14:54:00,2018-06-21 15:34:32,2018-07-04,1,009c09f439988bc06a93d6b8186dce73,89a51f50b8095ea78d5768f34c13a76f,2018-06-18 18:59:02,132.40,14.05,ccafc1c3f270410521c3c6f3b249870f,74820,goiania,GO,0.106118,True
17,403b97836b0c04a622354cf531062e5f,738b086814c6fcc74b8cc583f8516ee3,delivered,2018-01-02 19:00:43,2018-01-02 19:09:04,2018-01-03 18:19:09,2018-01-20 01:38:59,2018-02-06,1,638bbb2a5e4f360b71f332ddfebfd672,c4af86330efa7a2620772227d2d670c9,2018-01-12 19:09:04,1299.00,77.45,6e26bbeaa107ec34112c64e1ee31c0f5,21381,rio de janeiro,RJ,0.059623,True
18,116f0b09343b49556bbad5f35bee0cdf,3187789bec990987628d7a9beb4dd6ac,delivered,2017-12-26 23:41:31,2017-12-26 23:50:22,2017-12-28 18:33:05,2018-01-08 22:36:36,2018-01-29,1,a47295965bd091207681b541b26e40a5,ea8482cd71df3c1969d7b9473ff13abc,2018-01-02 23:50:22,27.99,15.10,6087cfc70fd833cf2db637a5e6e9d76b,88780,imbituba,SC,0.539478,True
19,85ce859fd6dc634de8d2f1e290444043,059f7fc5719c7da6cbafe370971a8d70,delivered,2017-11-21 00:03:41,2017-11-21 00:14:22,2017-11-23 21:32:26,2017-11-27 18:28:00,2017-12-11,1,cce679660c66e6fbd5c8091dfd29e9cd,d2374cbcbb3ca4ab1086534108cc3ab7,2017-11-29 00:14:22,17.90,11.85,d0ff1a7468fcc46b8fc658ab35d2a12c,13186,hortolandia,SP,0.662011,True


In [172]:
order_completed.iloc[0, 1]


'9ef432eb6251297304e76186b10a928d'

### 5.11 Análise de Valores Nulos

Verifiquei quais colunas têm nulos — contagem absoluta e percentual em relação ao total de linhas.

In [173]:
total_nan = order_completed.isna().sum()
print(f"\nTotal de nulos:\n {total_nan}\n")



Total de nulos:
 order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                  15
order_delivered_carrier_date     1194
order_delivered_customer_date    2454
order_estimated_delivery_date       0
order_item_id                       0
product_id                          0
seller_id                           0
shipping_limit_date                 0
price                               0
freight_value                       0
customer_unique_id                  0
customer_zip_code_prefix            0
customer_city                       0
customer_state                      0
freight_ratio                       0
is_on_time                          0
dtype: int64



In [174]:
percent_nan = order_completed.isna().mean() * 100
print(f"\nPercentual:\n {percent_nan}%")



Percentual:
 order_id                         0.000000
customer_id                      0.000000
order_status                     0.000000
order_purchase_timestamp         0.000000
order_approved_at                0.013316
order_delivered_carrier_date     1.059920
order_delivered_customer_date    2.178429
order_estimated_delivery_date    0.000000
order_item_id                    0.000000
product_id                       0.000000
seller_id                        0.000000
shipping_limit_date              0.000000
price                            0.000000
freight_value                    0.000000
customer_unique_id               0.000000
customer_zip_code_prefix         0.000000
customer_city                    0.000000
customer_state                   0.000000
freight_ratio                    0.000000
is_on_time                       0.000000
dtype: float64%


### Decisão — Valores Nulos nas Datas

Não preenchi nem removi os nulos das colunas de data.

- `order_approved_at`: 15 nulos — pedidos cancelados antes da aprovação
- `order_delivered_carrier_date`: 1194 nulos — pedidos não enviados
- `order_delivered_customer_date`: 2454 nulos — pedidos não entregues

Esses nulos fazem sentido — substituir por datas fictícias ia distorcer qualquer análise temporal. Nas análises que precisarem de data de entrega, filtro os nulos pontualmente.